# COMP6012 — Assignment 2 
## Evaluation of LLM Models for Text Classification and Generation


**Name:** Victor Tarus
**Student ID:** A1985890




## Step 1: Prerequisites

In [1]:
!pip install -q "transformers<5" datasets evaluate rouge_score scikit-learn pandas matplotlib torch torchvision
print("Done installing!")


Done installing!


In [ ]:
# Step 2
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

device = 0 if torch.cuda.is_available() else -1   # 0 = use GPU, -1 = use CPU
print("GPU available:", torch.cuda.is_available())


GPU available: False


## Part A: Text Classification (Yelp Reviews)

**Todo:** given a restaurant/business review, predict how many stars (1–5) it got.
load the Yelp dataset, take a small sample of it (to keep things fast), and try two different
pretrained models on it.

### Step 3: Load the Yelp dataset


In [3]:
yelp = load_dataset("Yelp/yelp_review_full")

print(yelp["test"][0])


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

### Step 4: Sampling

In [ ]:
SAMPLES_PER_STAR = 200   

def get_balanced_sample(dataset_split, n_per_class, seed):
    df = dataset_split.to_pandas()
    sample = df.groupby("label", group_keys=False).apply(
        lambda x: x.sample(n=n_per_class, random_state=seed)
    )
    return sample.sample(frac=1, random_state=seed).reset_index(drop=True)  

yelp_sample = get_balanced_sample(yelp["test"], SAMPLES_PER_STAR, SEED)
print("Sample size:", len(yelp_sample))
yelp_sample.head()


### Step 5: Load two pretrained models


In [ ]:
from transformers import pipeline

#Model 1: trained specifically to output 1-5 star predictions
model1 = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
    truncation=True,
)

#test
print(model1("The food was amazing and the staff were so friendly!"))


In [ ]:
# Model 2:general purpose model
model2 = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

star_labels = ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]
#test
print(model2("The food was amazing and the staff were so friendly!", star_labels))


### Step 6: Run both models on sample

In [ ]:
def predict_with_model1(review_texts):
    predictions = []
    for review in review_texts:
        result = model1(review[:512])[0]    
        stars = int(result["label"].split()[0])    
        predictions.append(stars)
    return predictions

def predict_with_model2(review_texts):
    predictions = []
    for review in review_texts:
        result = model2(review[:512], star_labels)
        top_label = result["labels"][0]         
        predictions.append(int(top_label.split()[0]))
    return predictions

reviews = yelp_sample["text"].tolist()
true_stars = (yelp_sample["label"] + 1).tolist() 

print("Running Model 1...")
model1_predictions = predict_with_model1(reviews)

print("Running Model 2 (this one is slower)...")
model2_predictions = predict_with_model2(reviews)

print("Done!")


### Step 7: Score the models


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def score_model(true_values, predicted_values, model_name):
    acc = accuracy_score(true_values, predicted_values)
    f1 = f1_score(true_values, predicted_values, average="macro")
    print(f"{model_name}: accuracy = {acc:.2f}, macro-F1 = {f1:.2f}")
    return {"model": model_name, "accuracy": acc, "macro_f1": f1}

score1 = score_model(true_stars, model1_predictions, "Sentiment BERT")
score2 = score_model(true_stars, model2_predictions, "Zero-shot BART")

classification_results = pd.DataFrame([score1, score2])
classification_results


In [ ]:

classification_results.set_index("model")[["accuracy", "macro_f1"]].plot(kind="bar", figsize=(6,4))
plt.title("Classification: Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("classification_comparison.png", dpi=150)
plt.show()


### Step 8 : Fairness check

In [ ]:
results_table = pd.DataFrame(
    {
        "true_stars": true_stars,
        "model1_prediction": model1_predictions,
        "model2_prediction": model2_predictions,
    }
)
results_table["model1_correct"] = (
    results_table["true_stars"] == results_table["model1_prediction"]
)
results_table["model2_correct"] = (
    results_table["true_stars"] == results_table["model2_prediction"]
)

accuracy_per_star = results_table.groupby("true_stars")[
    ["model1_correct", "model2_correct"]
].mean()
print(accuracy_per_star)


accuracy_per_star_named = accuracy_per_star.rename(
    columns={
        "model1_correct": "Sentiment BERT",
        "model2_correct": "Zero-shot BART",
    }
)

from matplotlib.ticker import PercentFormatter

ax = accuracy_per_star_named.plot(kind="bar", figsize=(6, 4))
plt.title("Accuracy by True Star Rating")
plt.xlabel("True Star Rating")
plt.ylabel("Accuracy (%)")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))  # show y-axis as percentages
plt.legend(title="Model")
plt.tight_layout()
plt.savefig("accuracy_by_star.png", dpi=150)
plt.show()

## Part B: Text Generation / Summarization (CNN/DailyMail)

**Todo:** given a news article, generate a short summary of it.



### Step 9: Load the CNN/DailyMail dataset and take a small sample

In [ ]:
cnn_dailymail = load_dataset("abisee/cnn_dailymail", "3.0.0")
print("ARTICLE (first 500 characters):")
print(cnn_dailymail["test"][0]["article"][:500])
print("\nHUMAN-WRITTEN SUMMARY:")
print(cnn_dailymail["test"][0]["highlights"])


In [ ]:
N_ARTICLES = 20  
news_sample = cnn_dailymail["test"].shuffle(seed=SEED).select(range(N_ARTICLES))
articles = news_sample["article"]
reference_summaries = news_sample["highlights"]    # the "correct answer" summaries written by humans
print(f"Loaded {len(articles)} articles")


### Step 10: Load two summarization models

- **BART-large-CNN**: pretrained specifically for summarizing news articles 
- **T5-small**: a smaller, general-purpose text-to-text model


In [ ]:
summarizer1 = pipeline("summarization", model="facebook/bart-large-cnn", device=device)
summarizer2 = pipeline("summarization", model="t5-small", device=device)
def truncate_words(text, max_words=700):
    words = text.split()
    return " ".join(words[:max_words])

GEN_KWARGS = dict(
    max_length=130,
    min_length=30,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    truncation=True,  
)

test_article = truncate_words(articles[0])
print("BART summary:", summarizer1(test_article, **GEN_KWARGS))
print("\nT5 summary:", summarizer2("summarize: " + test_article, **GEN_KWARGS))


### Step 11: Generate summaries for our whole sample


In [ ]:
def generate_summaries(summarizer, texts, prefix=""):
    summaries = []
    for text in texts:
        safe_text = truncate_words(text)     # <-- pre-truncate every article, not just the test one
        result = summarizer(prefix + safe_text, **GEN_KWARGS)
        summaries.append(result[0]["summary_text"])
    return summaries

print("Generating BART summaries...")
bart_summaries = generate_summaries(summarizer1, articles)

print("Generating T5 summaries...")
t5_summaries = generate_summaries(summarizer2, articles, prefix="summarize: ")

print("Done!\n")


print("=" * 80)
for i in range(3):
    print(f"\n--- Example {i} ---")
    print("REFERENCE:", reference_summaries[i])
    print("\nBART:     ", bart_summaries[i])
    print("\nT5:       ", t5_summaries[i])
print("=" * 80)


### Step 12:Score the summaries with ROUGE


In [ ]:
import evaluate
rouge = evaluate.load("rouge")

bart_scores = rouge.compute(predictions=bart_summaries, references=reference_summaries)
t5_scores = rouge.compute(predictions=t5_summaries, references=reference_summaries)

print("BART scores:", bart_scores)
print("T5 scores:  ", t5_scores)

generation_results = pd.DataFrame([
    {"model": "BART-large-CNN", **bart_scores},
    {"model": "T5-small", **t5_scores},
])
generation_results


In [ ]:
generation_results.set_index("model")[["rouge1", "rouge2", "rougeL"]].plot(kind="bar", figsize=(6,4))
plt.title("Summarization: Model Comparison (ROUGE)")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig("generation_comparison.png", dpi=150)
plt.show()


## Step 13:Save results

In [ ]:
classification_results.to_csv("classification_results.csv", index=False)
generation_results.to_csv("generation_results.csv", index=False)
accuracy_per_star.to_csv("accuracy_by_star.csv")

